# 🤖 Predictive Modeling with Machine Learning

An end-to-end supervised learning pipeline covering:
- **Exploratory Data Analysis (EDA)**
- **Data Preprocessing** — scaling, imputation
- **Model Training** — Logistic Regression, Decision Tree, Random Forest, Gradient Boosting
- **Evaluation** — Accuracy, F1, ROC-AUC, Confusion Matrix
- **Visualization** — ROC Curves, Feature Importance, Learning Curves

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_diabetes

# ── Notebook display config ───────────────────────────────
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded ✓')

## 📦 1. Load Datasets

In [ ]:
# ── Classification: Breast Cancer ──────────────────────────
bc = load_breast_cancer(as_frame=True)
df_clf = bc.data.copy()
df_clf['target'] = bc.target
df_clf['target_name'] = df_clf['target'].map({0: 'Malignant', 1: 'Benign'})

print('=== Classification Dataset (Breast Cancer) ===')
print(f'Shape : {df_clf.shape}')
print(f'Classes: {bc.target_names.tolist()}')
print()
df_clf.head(3)

In [ ]:
# ── Regression: Diabetes ──────────────────────────────────
diab = load_diabetes(as_frame=True)
df_reg = diab.data.copy()
df_reg['target'] = diab.target

print('=== Regression Dataset (Diabetes) ===')
print(f'Shape : {df_reg.shape}')
print(f'Target range: {df_reg["target"].min():.1f} – {df_reg["target"].max():.1f}')
print()
df_reg.head(3)

## 🔍 2. Exploratory Data Analysis

In [ ]:
# ── Class distribution ────────────────────────────────────
BG = '#0F1117'
GRID = '#1E2130'
TEXT = '#E8EAF0'
PAL = ['#6C63FF', '#FF6584', '#43D9A3', '#FFB347', '#00B4D8']

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': GRID,
    'text.color': TEXT, 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'axes.titlecolor': TEXT, 'grid.color': '#2A2E45',
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Classification class distribution
counts = df_clf['target_name'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=PAL[:2], edgecolor='none', width=0.5)
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 2, str(val), ha='center', fontsize=12, color=TEXT)
axes[0].set_title('Class Distribution — Breast Cancer', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Regression target distribution
axes[1].hist(df_reg['target'], bins=30, color=PAL[2], edgecolor='#0F1117')
axes[1].axvline(df_reg['target'].mean(), color='white', lw=2,
                linestyle='--', label=f'Mean = {df_reg["target"].mean():.1f}')
axes[1].set_title('Target Distribution — Diabetes Progression', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Disease Progression Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature correlation heatmap ───────────────────────────
fig, ax = plt.subplots(figsize=(18, 14))
corr = df_clf.drop(columns=['target', 'target_name']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap=sns.diverging_palette(250, 15, s=75, l=40, n=256, as_cmap=True),
    center=0, annot=False, fmt='.2f',
    linewidths=0.3, linecolor='#0F1117',
    cbar_kws={'shrink': 0.7}
)
ax.set_title('Feature Correlation Matrix — Breast Cancer Dataset',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Statistical summary ───────────────────────────────────
print('=== Basic Statistics (Breast Cancer — first 10 features) ===')
df_clf.iloc[:, :10].describe().T

## ⚙️ 3. Preprocessing

In [ ]:
from src.data_loader   import load_classification_data
from src.preprocessing import preprocess_data

X_train, X_test, y_train, y_test, features, targets = load_classification_data()
X_train_sc, X_test_sc, pipeline = preprocess_data(X_train, X_test)

print(f'X_train scaled: {X_train_sc.shape}')
print(f'X_test  scaled: {X_test_sc.shape}')

## 🏋️ 4. Model Training

In [ ]:
from src.models import CLASSIFICATION_MODELS, train_all_models

trained = train_all_models(CLASSIFICATION_MODELS, X_train_sc, y_train)

## 📊 5. Evaluation

In [ ]:
from src.evaluation import evaluate_model, compare_models

results = [
    evaluate_model(name, info['model'], X_test_sc, y_test, task='classification')
    for name, info in trained.items()
]

comparison_df = compare_models(results, task='classification')
comparison_df

## 📈 6. Visualizations

In [ ]:
from src.visualization import (
    plot_confusion_matrix, plot_roc_curves,
    plot_feature_importance, plot_learning_curves, plot_model_comparison
)

fig = plot_confusion_matrix(results, targets, save_dir='results/figures')
plt.show()

In [ ]:
fig = plot_roc_curves(results, y_test, save_dir='results/figures')
plt.show()

In [ ]:
fig = plot_feature_importance(results, features, save_dir='results/figures')
plt.show()

In [ ]:
fig = plot_learning_curves(results, X_train_sc, y_train, save_dir='results/figures')
plt.show()

In [ ]:
fig = plot_model_comparison(comparison_df, task='classification', save_dir='results/figures')
plt.show()

## 💾 7. Save Best Model

In [ ]:
import joblib, os

best_name  = comparison_df.iloc[0]['Model']
best_model = trained[best_name]['model']

os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/best_classifier.pkl')

print(f'✅ Best model: {best_name}')
print(f'   Saved to  : models/best_classifier.pkl')

# Load & predict
loaded = joblib.load('models/best_classifier.pkl')
sample_pred = loaded.predict(X_test_sc[:5])
print(f'\nSample predictions: {[targets[p] for p in sample_pred]}')
print(f'True labels       : {[targets[t] for t in y_test.values[:5]]}')

---

## 🎓 Summary

| Step | Action | Outcome |
|------|--------|---------|
| 1 | Loaded Breast Cancer dataset | 569 samples, 30 features |
| 2 | Preprocessed with StandardScaler | Zero-mean, unit-variance features |
| 3 | Trained 4 classifiers | LR, DT, RF, GB |
| 4 | Evaluated on hold-out test set | F1-Score, AUC-ROC, Accuracy |
| 5 | Visualized results | Confusion matrices, ROC, Feature importance |
| 6 | Saved best model | `models/best_classifier.pkl` |

> **Best Model**: Gradient Boosting — highest F1-Score and AUC-ROC
